ChatGPT API를 이용한 txt 파일 요약 과정

In [1]:
! pip install --upgrade google-generativeai


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
! pip show google-generativeai

Name: google-generativeai
Version: 0.8.6
Summary: Google Generative AI High level API client library and tools.
Home-page: https://github.com/google/generative-ai-python
Author: Google LLC
Author-email: googleapis-packages@google.com
License: Apache 2.0
Location: C:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages
Requires: google-ai-generativelanguage, google-api-core, google-api-python-client, google-auth, protobuf, pydantic, tqdm, typing-extensions
Required-by: 


In [3]:
import google.generativeai as genai
import os
import glob
import pandas as pd
from tqdm import tqdm

# API 키 설정 (환경 변수 사용)
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
model = genai.GenerativeModel("gemini-pro")

# --------------

In [4]:
def load_texts_from_folder(folder_path):
    texts = []

    for file in glob.glob(os.path.join(folder_path, "*.txt")):
        with open(file, encoding="utf-8") as f:
            content = f.read().strip()
            if content:
                texts.append(f"[파일: {os.path.basename(file)}]\n{content}")

    return "\n\n".join(texts)

In [5]:
def split_text(text, max_len=1200):
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    chunks, current = [], ""

    for line in lines:
        if len(current) + len(line) <= max_len:
            current += " " + line
        else:
            chunks.append(current.strip())
            current = line

    if current:
        chunks.append(current.strip())

    return chunks

In [7]:
base_dir = "변환됨"
folders = [
    "국기연_발간물",
    "국방위_국정감사",
    "국방위_보도",
    "국방위_회의록",
    "국방부_정책자료"
]

for folder in folders:
    folder_path = os.path.join(base_dir, folder)
    full_text = load_texts_from_folder(folder_path)
    chunks = split_text(full_text)

    results = []

    for i, chunk in enumerate(tqdm(chunks, desc=folder)):
        summary = split_text(chunk)

        if "의미 있는 정보 없음" in summary:
            continue

        results.append({
            "chunk_id": i,
            "clean_summary": summary
        })

    df = pd.DataFrame(results)
    df.to_csv(f"{folder}_summary.csv", index=False, encoding="utf-8-sig")

    print(f"✅ {folder}_summary.csv 생성 완료")

국기연_발간물: 100%|██████████| 2328/2328 [00:00<00:00, 578216.36it/s]


✅ 국기연_발간물_summary.csv 생성 완료


국방위_국정감사: 100%|██████████| 872/872 [00:00<00:00, 866895.73it/s]


✅ 국방위_국정감사_summary.csv 생성 완료


국방위_보도: 100%|██████████| 32/32 [00:00<?, ?it/s]


✅ 국방위_보도_summary.csv 생성 완료


국방위_회의록: 100%|██████████| 3407/3407 [00:00<00:00, 558159.27it/s]


✅ 국방위_회의록_summary.csv 생성 완료


국방부_정책자료: 100%|██████████| 119/119 [00:00<?, ?it/s]

✅ 국방부_정책자료_summary.csv 생성 완료


# -----------------------
- 회의록만 요약하기

In [ ]:
import google.generativeai as genai
import os
import glob
import pandas as pd
from tqdm import tqdm

# API 키 설정 (환경 변수 사용)
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
model = genai.GenerativeModel("gemini-pro")

In [ ]:
def generate_content(chunks):
    joined = "\n".join(chunks)

    return f"""
다음은 대한민국 국회 국방위원회 회의록 일부이다.
정치적 수사는 모두 제거하고, 국방 연구과제·기술·결정사항 및 외국과의 기술 계약 중심으로 요약하라.

요약 조건:
- 핵심 쟁점 3~5개
- 국방 연구과제·기술·결정사항 및 외국과의 기술 계약 관련 내용 우선
- 추측이나 해석 금지

회의록:
{joined}
"""

In [4]:
def call_gemini(prompt: str) -> str:
    response = model.generate_content(prompt)
    return response.text.strip()